In [21]:
%pip install -q --upgrade pip

# Install a ragas version that supports langchain-core >=0.3,.
# plus a compatible langchain-community pin to avoid the ChatVertexAI import error.
%pip install -q \
    "ragas>=0.2.15" \
    "langchain-google-genai>=2.0.0" \
    "langchain-community<0.4.2" \
    langchain_cohere \
    langchain_core \
    litellm \
    datasets pandas matplotlib seaborn

In [2]:
import os
import asyncio
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from tqdm.auto import tqdm

import cohere
from ragas.llms import llm_factory
from ragas.embeddings import LiteLLMEmbeddings   # universal embedder, supports Cohere
from ragas.metrics.collections import (
    SummaryScore,
    SemanticSimilarity,
    AnswerCorrectness,
)

In [4]:
from getpass import getpass

COHERE_API_KEY = getpass("Enter your Cohere API key: ")

In [5]:
ds = load_dataset("pameydorke/redred-gemma-4-E2B-it-lora-summaries", split="train").select(range(1))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [34]:
# cohere_client = cohere.ClientV2(api_key=COHERE_API_KEY)
cohere_async_client = cohere.AsyncClientV2(api_key=COHERE_API_KEY)

In [36]:
import instructor

patched_cohere_client = instructor.from_cohere(
    cohere_async_client,
    mode=instructor.Mode.COHERE_TOOLS,
)

In [37]:
ragas_llm = llm_factory(
    model="command-r-plus",
    provider="cohere",
    client=patched_cohere_client,
)

In [39]:
ragas_embeddings = LiteLLMEmbeddings(
    model="cohere/embed-english-v3.0",
    api_key=COHERE_API_KEY,
)

In [40]:
summarization_scorer = SummaryScore(llm=ragas_llm)
semantic_sim_scorer = SemanticSimilarity(embeddings=ragas_embeddings)
answer_correctness_scorer = AnswerCorrectness(
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

In [41]:
async def evaluate_responses(dataset, response_column: str) -> pd.DataFrame:
    """
    Evaluate every sample in `dataset` using the given response column
    (e.g. "base_response" or "ft_response").
    Returns a DataFrame with one row per sample and three metric columns.
    """
    rows = []
    for sample in tqdm(dataset, desc=f"Evaluating {response_column}"):
        user_input = sample["user_input"]
        reference  = sample["reference"]
        response   = sample[response_column]

        # --- Summarization Score ---
        # Needs the original text as a list of reference contexts
        sum_result = await summarization_scorer.ascore(
            reference_contexts=[user_input],
            response=response,
        )

        # --- Semantic Similarity ---
        sim_result = await semantic_sim_scorer.ascore(
            reference=reference,
            response=response,
        )

        # --- Answer Correctness ---
        # Requires user_input, response, and reference
        ans_result = await answer_correctness_scorer.ascore(
            user_input=user_input,
            response=response,
            reference=reference,
        )

        rows.append({
            "summarization_score": sum_result.value,
            "semantic_similarity": sim_result.value,
            "answer_correctness": ans_result.value,
        })

    return pd.DataFrame(rows)

In [42]:
base_scores = await evaluate_responses(ds, "base_response")

Evaluating base_response:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:instructor.v2.retry:API call failed on attempt 1: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': '4e322dbcb70d536df1f154876eea1fc4', 'x-endpoint-monthly-call-limit': '1000', 'x-trial-endpoint-call-limit': '20', 'x-trial-endpoint-call-remaining': '19', 'date': 'Wed, 23 Sep 2026 14:54:13 GMT', 'x-envoy-upstream-service-time': '14', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 404, body: {'id': 'c45928ca-fa39-4a02-bbac-463b3e7bb210', 'message': "model 'command-r-plus' was removed on September 15, 2025. See https://docs.cohere.com/docs/models#command for a list of models you 

InstructorRetryException: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': '4e322dbcb70d536df1f154876eea1fc4', 'x-endpoint-monthly-call-limit': '1000', 'x-trial-endpoint-call-limit': '20', 'x-trial-endpoint-call-remaining': '19', 'date': 'Wed, 23 Sep 2026 14:54:13 GMT', 'x-envoy-upstream-service-time': '14', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 404, body: {'id': 'c45928ca-fa39-4a02-bbac-463b3e7bb210', 'message': "model 'command-r-plus' was removed on September 15, 2025. See https://docs.cohere.com/docs/models#command for a list of models you can use instead."}

### prev

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from ragas import evaluate, EvaluationDataset 
from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
from ragas.dataset_schema import SingleTurnSample
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_cohere import ChatCohere
from ragas.llms import LangchainLLMWrapper
from getpass import getpass

/tmp/ipykernel_65158/2074089641.py:7: DeprecationWarning: Importing SummarizationScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SummarizationScore
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_65158/2074089641.py:7: DeprecationWarning: Importing SemanticSimilarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SemanticSimilarity
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_65158/2074089641.py:7: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import Summari

In [5]:
COHERE_API_KEY = getpass("Enter your Cohere API key: ")

In [4]:
GOOGLE_API_KEY = getpass("Enter your GOOGLE API key: ")

In [ ]:
# cohere_llm = ChatCohere(
#     cohere_api_key=COHERE_API_KEY,
#     model="command-a-03-2025",
#     temperature=0,
#     max_tokens=4096,          # RAGAS rubrics can be verbose
# )

In [6]:
import cohere

cohere_client = cohere.ClientV2(COHERE_API_KEY)

In [7]:
from ragas.llms import llm_factory

judge_llm = llm_factory(
    "command-a-03-2025",           # model name
    provider="cohere",             # tell RAGAS this is Cohere
    client=cohere_client,          # pass the raw Cohere client
    adapter="litellm",             # <-- the key: force LiteLLM adapter
    temperature=0,
    max_tokens=8192,
)

In [8]:
# embeddings = LangchainEmbeddingsWrapper(
#     GoogleGenerativeAIEmbeddings(
#         model="gemini-embedding-001",  # or "models/embedding-001"
#         google_api_key=GOOGLE_API_KEY,
#     )
# )

from google import genai
from ragas.embeddings import GoogleEmbeddings

client = genai.Client(api_key=GOOGLE_API_KEY)

# 2a. Option A — GoogleEmbeddings directly (explicit client)
gemini_embeddings = GoogleEmbeddings(
    client=client,
    model="gemini-embedding-001",   # or "text-embedding-004"
)

In [9]:
ds = load_dataset("pameydorke/redred-gemma-4-E2B-it-lora-summaries", split="train").select(range(1))
df = ds.to_pandas()

# Show the columns and a sample row
print(df.columns.tolist())
df.head(2)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


['user_input', 'reference', 'base_response', 'ft_response']


,user_input,reference,base_response,ft_response
0,Original Post: Help with Small living room Use...,The OP asked about general design suggestions ...,"The original poster, moving into a 1920s craft...",The user is asking for advice on how to arrang...


In [11]:
def build_samples(row, response_col):
    """Create a SingleTurnSample for Ragas from a dataframe row."""
    return SingleTurnSample(
        user_input=row["user_input"],          # normalized Reddit thread text
        response=row[response_col],            # model summary (base or ft)
        reference=row["reference"],            # human‑made summary
        # SummarizationScore needs the original context to extract keyphrases
        reference_contexts=[row["user_input"]],
    )

# Build sample lists for the base model and the fine‑tuned model
base_samples = [build_samples(row, "base_response") for _, row in df.iterrows()]
ft_samples   = [build_samples(row, "ft_response")   for _, row in df.iterrows()]

base_dataset = EvaluationDataset(samples=base_samples)
ft_dataset = EvaluationDataset(samples=ft_samples)

In [12]:
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextPrecision

# Instantiate each metric with the judge LLM
faithfulness_metric = Faithfulness(llm=judge_llm)
answer_relevancy_metric = AnswerRelevancy(llm=judge_llm, embeddings=gemini_embeddings)
context_precision_metric = ContextPrecision(llm=judge_llm)

In [16]:
async def score_sample(sample):
    return {
        "faithfulness": (await faithfulness_metric.ascore(
            user_input=sample["question"],
            response=sample["answer"],
            retrieved_contexts=sample["contexts"],
        )).value,
        "answer_relevancy": (await answer_relevancy_metric.ascore(
            user_input=sample["question"],
            response=sample["answer"],
        )).value,
        "context_precision": (await context_precision_metric.ascore(
            user_input=sample["question"],
            retrieved_contexts=sample["contexts"],
            reference=sample["ground_truth"],
        )).value,
    }

In [17]:
import asyncio

async def run_evaluation(dataset):
    results = []
    for sample in dataset:
        # Add a small delay to respect rate limits
        await asyncio.sleep(1) 
        try:
            scores = await score_sample(sample)
            results.append({**sample, **scores})
        except Exception as e:
            print(f"Error scoring sample: {e}")
            results.append({**sample, "error": str(e)})
    return results

In [18]:
scored_results = await run_evaluation(ds)

Error scoring sample: 'question'


In [15]:
metrics = [
    faithfulness_metric,
    answer_relevancy_metric,
    context_precision_metric,
]

base_result = evaluate(
    dataset=base_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=gemini_embeddings,
)

TypeError: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]

In [ ]:
ft_result = evaluate(
    dataset=ft_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=embeddings,
)

# Convert results to dataframes for easy inspection
base_df = base_result.to_pandas()
ft_df   = ft_result.to_pandas()

print("Base model scores:")
print(base_df.mean(numeric_only=True))
print("\nFine‑tuned model scores:")
print(ft_df.mean(numeric_only=True))